In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:

data = spark.table("electronics_retailer_clg.bronze.products")

from pyspark.sql import functions as F

def normalize_prizes(df, col_names):
    for col in col_names:
        df = df.withColumn(
            col, 
            F.regexp_replace(F.col(col), '[^0-9.]', '').cast('double')
        )
    return df

data = normalize_prizes(data, ['unit_cost_usd', 'unit_price_usd'])
display(data)



In [0]:
data = data.withColumn("productkey", F.trim(F.col("productkey")).cast('int'))

In [0]:
data.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("electronics_retailer_clg.silver.products")

In [0]:
# from pyspark.sql.functions import col, trim, regexp_replace


# df = spark.table("electronics_retailer_clg.bronze.products")
# display(df)


# df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])

# for c in df.columns:
#     df = df.withColumn(c, trim(col(c)))


# df = df.withColumn(
#     "unit_price_usd",
#     regexp_replace("unit_price_usd", "\\$", "")
# )

# df = df.withColumn(
#     "unit_price_usd",
#     regexp_replace("unit_price_usd", ",", "")
# )

# df = df.withColumn("productkey", col("productkey").cast("int")) \
#        .withColumn("unit_price_usd", col("unit_price_usd").cast("double"))


# df = df.select(
#     "productkey",
#     "category",
#     "unit_price_usd"
# )

# # display(df)

# df.write.format("delta") \
#     .mode("overwrite") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("electronics_retailer_clg.silver.products")

# print("Products cleaned successfully (price issue fixed)")